# BP7 Gate 4 — Statistical Validation & Explainability
**Customer360 Navigator Enterprise Suite — Customer360 Navigator Decision Engine**

## Why this gate looks different from BP1-3's own Gate 4 (and different again from BP5's/BP6's own adaptations)
The Master Plan's generic Gate 4 row (Section 8's Gate table) reads: output = *"Bootstrap CI,
calibration, confusion matrix, SHAP sample"*; exit criteria = *"All checks numeric and
reproducible; leakage re-confirmed"*; compliance touchpoint = *"Independent-style validation
record (SR 11-7 second-line analog); disparate-impact check where applicable (ECOA/Reg B)"*. That
row assumes a supervised classifier with predicted probabilities and a labeled ground-truth column
to validate. BP7 has neither, at any gate — Gate 1's own real, live-run `policy.json` is explicit
that `customer360_priority_decision` is "NOT a single trained ML target", and Gate 3's own
`benchmark_candidate()` docstring already disclosed "there is no labeled `customer360_priority_
decision` column in the real CFPB extract". Mirroring BP5's and BP6's own non-classic-classifier
Gate 4 adaptations, this gate maps each generic output onto a real BP7 analog rather than
fabricating an accuracy number that does not exist:

- **"Bootstrap CI"** → a real percentile bootstrap 95% CI (1,000 resamples, seed = this BP's own
  `random_state`) around two real champion-output statistics: `intervention_flag_rate` and
  `bp3_agreement_rate` — both real, live-computed on the champion's real, full-population score,
  never a point estimate presented as exact.
- **"calibration"** → no predicted probability exists to calibrate against a real outcome (BP7's
  `priority_score` is a deterministic weighted combination, not a probabilistic forecast fit to
  minimize a loss). This gate instead independently **reproduces** Gate 3's own recorded champion
  numbers from scratch, in this notebook's own fresh run, using the champion's own real weights
  read live from `configs/bp7_customer_navigator_decision_engine.yaml` (never hardcoded) — and
  asserts the reproduction matches Gate 3's recorded `gate3_benchmark_results.csv` row within
  floating-point tolerance, the same "independent reproduction" role BP6 Gate 4 used calibration's
  slot for.
- **"confusion matrix"** → a real 2×2 agreement/reference cross-tab of `intervention_flag`
  (True/False) × BP3's own already-validated `bp3_predicted_label` (1/0) — honestly labeled as a
  coherence/reference table, **never** a true confusion matrix, since there is no labeled ground
  truth for `customer360_priority_decision` to score against (the identical limitation Gate 3's
  own `bp3_agreement_rate` metric already disclosed).
- **"SHAP sample"** → BP7's weighted-linear-combination rule is already a fully transparent,
  deterministic formula — real explainability here means an **exact** per-row decomposition of
  `priority_score` into its three additive terms (`contribution_bp2` / `contribution_bp3` /
  `contribution_bp4`, which sum to `priority_score` exactly, verified live by a real
  reconstruction-error check), not an approximation of feature importance for a model that does
  not exist. Disclosed explicitly as a genuine strength over SHAP here, not a lesser substitute.
- **"leakage re-confirmed"** → a fresh, live re-check that none of Gate 1's own `BARRED_COLUMNS`
  (raw `Company response to consumer`, `Timely response?`, `Date received`, `Date sent to
  company`, `Tags`) are present on the real scoring Gold layer
  (`cfpb_decision_engine_context_gold.parquet`) this gate reloads — never trusted from Gate
  1/2/3's own prior claim.
- **"disparate-impact check where applicable (ECOA/Reg B)"** → Section 12 below. This is the real
  substance of this gate's compliance requirement, not a checkbox — see the next section.

## The real question this gate had to resolve first: can Gate 3's honest deferral be closed for real?
Gate 3's own real, live check (`check_disparate_impact_carry_forward`) found that BP7's own Gate 2
Gold layer does not carry BP3's `tags_group` grouping (it lives only in BP3's own held-out
test-split `gate5_decision_records.csv`, keyed by a `row_index` local to that split, not
`Complaint ID`), and honestly deferred the disparate-impact-style check on BP7's own champion
`intervention_flag` — grouped the same way BP3's own real, already-closed governance investigation
grouped it — to this gate, rather than reconstructing `tags_group` at Gate 3 from the raw, barred
`Tags` column as a Gate 3 **input**. This gate re-investigated that deferral for real (never
guessed past it):

1. **BP7 Gate 1's own real `policy.json` (`compliance_touchpoint.ecoa_reg_b`)**, re-read live
   below, explicitly commits — in the same paragraph that states `'Tags' ... barred from every BP7
   input` — to *"re-running a disparate-impact-style check on BP7's own final priority_score/
   intervention_flag, grouped by the same tags_group dimension, at BP7's own Gate 4 — not a legal
   determination of ECOA/Reg B compliance, a monitoring signal for a human reviewer, exactly BP3's
   own stated limitation."* Gate 1 itself already drew the distinction this gate needed: "barred
   from every BP7 **input**" means barred as an input to the `priority_score` computation, not
   barred from a downstream, decision-blind audit/monitoring use that only groups
   *already-computed* outputs.
2. **BP3's own real, already-closed Gate 4** (`bp3_complaint_escalation_prediction_g4_statistical_
   validation_explainability.ipynb`) already drew that identical distinction for itself: `Tags` was
   barred from BP3's own feature set at Gate 1, yet BP3's own Gate 4 read it — *read-only, never a
   feature, never one-hot encoded, never fit on* — purely for its own post-hoc disparate-impact
   monitoring section (`feature_data["Tags"] = df_pl["Tags"].cast(pl.Utf8).fill_null("NO_TAG")
   .to_list()`), with `no_barred_column_in_feature_frame` still asserted true throughout. This is
   the exact real precedent Gate 1's own text points to ("exactly BP3's own stated limitation") —
   confirmed live below by re-reading BP3's own notebook source, not assumed from this markdown
   cell's own summary of it.
3. **Where the real data for it lives**, live-checked below: BP3's own real Gate 2 Gold layer
   (`data/processed/cfpb_intervention_escalation_gold.parquet`) turns out to already carry BOTH
   `Complaint ID` (unique, int64) AND a `Tags` passthrough column for the **full real population**
   (1,048,575 rows — not just BP3's own held-out test split), because BP3's own Gate 2 build read
   the raw CFPB extract directly and kept `Tags` as a passthrough column precisely so BP3's own
   Gate 4 could do the read-only check described above. This gate never reads the raw CFPB extract
   itself — it reads BP3's own already-real-run-confirmed, already-governance-reviewed Gold-layer
   artifact instead, one real, already-approved step removed from the raw column Gate 1 bars as a
   BP7 **input**.

**Conclusion (verified live below, not assumed here)**: the audit-only, non-scoring,
downstream-of-the-decision use of `tags_group` is legitimate and technically feasible for BP7 Gate
4, on the same real precedent and the same real statistical convention (selection-rate-based
adverse-impact ratio, EEOC four-fifths-rule) BP3's own closed investigation used. Section 12 below
builds it for real: `tags_group` is loaded from BP3's own real Gold layer, left-joined onto BP7's
own **already-scored** population via `Complaint ID` **strictly after** `priority_score`/
`intervention_flag`/`recommended_action` are computed — it never enters
`attach_normalized_signal_columns()` or `score_priority_rule()`, and Section 11's own fresh
leakage re-check confirms the scoring Gold layer itself still carries no `Tags` column at all. If
any one of the three findings above had come back differently (a barred-entirely reading of Gate
1's text, or BP3's own Gold layer missing `Complaint ID`/`Tags` for the full population), this
section would instead report an honest, specific non-fabricated deferral or decline here — exactly
as Gate 3 did — rather than a worked-around check.

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; you run it. Every real number below — every
  reproduced champion statistic, bootstrap CI, contribution decomposition, and disparate-impact
  ratio — is a real measurement from your own machine's run against your own real Gate 2/3
  artifacts and BP3's own real Gold layer, never simulated.
- **Zero-fabrication**: no financial-impact, illustrative, or assumption-based content anywhere.
  No BP1-6 notebook, module, or artifact is modified — this gate only *reads* BP3's own
  already-delivered Gold layer.
- **WARP**: `configure_performance()` first, before any heavy import, same as every other
  notebook.
- **HYPER**: every new Gate 4 function lives in `src/features/bp7_decision_engine_features.py`
  (extended, not duplicated) — reuses `attach_normalized_signal_columns()` /
  `score_priority_rule()` / `benchmark_candidate()` from this same module's own Gate 3 section
  **unmodified** for the independent-reproduction step, and reuses `src/utils/bp1_config_sync.py`
  unmodified (seventh BP-gate to do so).
- **Idempotent**: re-running this notebook overwrites this gate's own config block and artifacts
  in place; every other gate's block is preserved verbatim regardless of position. This gate does
  **not** modify `status` — this project's own established convention (BP7 Gate 2 and Gate 3 both
  left `status: "gate1_confirmed"` untouched) is that `status` is updated only at a BP's own final
  gate.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same project-root resolver as every other notebook.
- **No test coverage added here** — matching every other BP's own precedent, BP7's own first tests
  are deferred to its own Gate 6.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_reproduction_check.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_bootstrap_ci.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_bp3_agreement_crosstab.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_contribution_decomposition_summary.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_contribution_decomposition_sample.csv`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_leakage_reconfirmation.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_disparate_impact_audit.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_disparate_impact_breakdown.csv`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate4_statistical_validation_explainability_summary.json`
- `configs/bp7_customer_navigator_decision_engine.yaml` — Gate 4 marker block appended/overwritten
  (`status` untouched).

## Prerequisites
BP7 Gate 3 must have been real-run at least once — this notebook checks the Gate 3 config block
and `gate3_decision_rule_benchmark_summary.json`/`gate3_benchmark_results.csv` live and raises a
clear error if any is missing. It also reads BP3's own real Gold layer
(`data/processed/cfpb_intervention_escalation_gold.parquet`) and BP3's own real
`gate4_statistical_validation.json` live (both already real-run-confirmed) — never hardcoding
either BP's own numbers.

## If a structural check below fails
It raises `AssertionError` naming the failing check. This gate never relaxes the four-fifths-rule
convention, never reconstructs `tags_group` from the raw CFPB extract, and never substitutes a
fabricated number for a real one that could not be computed.

In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP7 Gate 4 statistical validation / explainability notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os
import sys
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
    memory_headroom_gb,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)
_ram_ceiling_fraction = RESOURCE_LIMITS["ceilings"]["max_ram_fraction"]
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb(_ram_ceiling_fraction)} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import yaml  # noqa: E402

from utils.bp1_config_sync import write_gate_block  # noqa: E402
from features.bp7_decision_engine_features import (  # noqa: E402
    CANDIDATE_NAMES,
    DEFAULT_INTERVENTION_THRESHOLD,
    N_DEFAULT_BOOTSTRAP,
    load_bp2_friction_ordinal_ranks,
    load_upstream_validated_metrics,
    compute_bp4_join_coverage,
    compute_bp2_bp3_correlation,
    attach_normalized_signal_columns,
    compute_candidate_raw_weights,
    normalize_candidate_weights,
    score_priority_rule,
    benchmark_candidate,
    fit_lr_diagnostic,
    select_champion,
    bootstrap_ci_for_rate,
    build_bp3_agreement_crosstab,
    compute_priority_score_contribution_decomposition,
    summarize_contribution_decomposition,
    reconfirm_no_barred_columns_in_gold_layer,
    KNOWN_TAGS_GROUPS,
    load_bp3_gold_tags_group,
    attach_tags_group_for_audit,
    compute_disparate_impact_audit,
)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
ARTIFACTS_DIR = NOTEBOOKS_DIR / "bp7_customer_navigator_decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

BP7_CONFIG_PATH = CONFIGS_DIR / "bp7_customer_navigator_decision_engine.yaml"
GOLD_PATH = DATA_PROCESSED / "cfpb_decision_engine_context_gold.parquet"
GATE3_SUMMARY_PATH = ARTIFACTS_DIR / "gate3_decision_rule_benchmark_summary.json"
GATE3_BENCHMARK_CSV_PATH = ARTIFACTS_DIR / "gate3_benchmark_results.csv"
BP7_POLICY_PATH = ARTIFACTS_DIR / "policy.json"

BP3_ARTIFACTS_DIR = NOTEBOOKS_DIR / "bp3_complaint_escalation_prediction" / "artifacts"
BP3_GOLD_PATH = DATA_PROCESSED / "cfpb_intervention_escalation_gold.parquet"
BP3_GATE4_STAT_VALIDATION_PATH = BP3_ARTIFACTS_DIR / "gate4_statistical_validation.json"

REPRODUCTION_CHECK_PATH = ARTIFACTS_DIR / "gate4_reproduction_check.json"
BOOTSTRAP_CI_PATH = ARTIFACTS_DIR / "gate4_bootstrap_ci.json"
AGREEMENT_CROSSTAB_PATH = ARTIFACTS_DIR / "gate4_bp3_agreement_crosstab.json"
CONTRIBUTION_SUMMARY_PATH = ARTIFACTS_DIR / "gate4_contribution_decomposition_summary.json"
CONTRIBUTION_SAMPLE_CSV_PATH = ARTIFACTS_DIR / "gate4_contribution_decomposition_sample.csv"
LEAKAGE_RECONFIRMATION_PATH = ARTIFACTS_DIR / "gate4_leakage_reconfirmation.json"
DISPARATE_IMPACT_AUDIT_PATH = ARTIFACTS_DIR / "gate4_disparate_impact_audit.json"
DISPARATE_IMPACT_BREAKDOWN_CSV_PATH = ARTIFACTS_DIR / "gate4_disparate_impact_breakdown.csv"
SUMMARY_JSON_PATH = ARTIFACTS_DIR / "gate4_statistical_validation_explainability_summary.json"

for p in (BP7_CONFIG_PATH, GOLD_PATH, GATE3_SUMMARY_PATH, GATE3_BENCHMARK_CSV_PATH, BP7_POLICY_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP7 Gate 3 has been real-run at least once."
        )

# ============================================================
# SECTION 4: Gate 3 prerequisite check (live) - never trusted from memory, re-read every run.
# ============================================================
with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    full_config_text = f.read()
full_config = yaml.safe_load(full_config_text)
gate3_block_marker = (
    "# --- Gate 3 (Decision-Rule-Scheme Benchmark & Champion Selection) results "
    "(appended, idempotent overwrite) ---"
)
gate3_block_present = gate3_block_marker in full_config_text
gate3_confirmed = (
    gate3_block_present
    and full_config.get("champion_rule_scheme") is not None
    and full_config.get("gold_layer_rows_written") is not None
)
assert gate3_confirmed, (
    "BP7 Gate 3 does not appear to have completed successfully (gate3_block_present="
    f"{gate3_block_present}, champion_rule_scheme={full_config.get('champion_rule_scheme')!r}). "
    "Run Gate 3 for real before Gate 4."
)
RANDOM_STATE = full_config.get("random_state", 42)
RECORDED_CHAMPION = full_config["champion_rule_scheme"]
print(f"[OK] Gate 3 prerequisite confirmed - recorded champion_rule_scheme={RECORDED_CHAMPION!r}.")

with open(GATE3_SUMMARY_PATH, "r", encoding="utf-8") as f:
    gate3_summary = json.load(f)
gate3_benchmark_df = pd.read_csv(GATE3_BENCHMARK_CSV_PATH)

# ============================================================
# SECTION 5: Load the real Gold layer live, confirm row count still matches Gate 2/3's own
# recorded count (never assumed to be unchanged since Gate 3 ran).
# ============================================================
gold_pl = pl.read_parquet(GOLD_PATH)
live_row_count = gold_pl.height
row_count_matches_config = live_row_count == full_config.get("gold_layer_rows_written")
print(
    f"[OK] Real Gold layer loaded live: {live_row_count:,} rows x {gold_pl.width} cols "
    f"(matches config's own recorded gold_layer_rows_written: {row_count_matches_config})."
)
assert row_count_matches_config, (
    "Real Gold layer row count no longer matches the config's own recorded "
    f"gold_layer_rows_written ({full_config.get('gold_layer_rows_written')}) - re-run Gate 2/3 "
    "before trusting Gate 4's own results below."
)

# ============================================================
# SECTION 6: INDEPENDENT REPRODUCTION ("calibration" slot analog) - re-derive Gate 3's entire
# champion-selection pipeline from scratch, in THIS notebook's own fresh run, reusing Gate 3's own
# functions UNMODIFIED (HYPER) against the same real Gold layer - never by reading Gate 3's own
# config-recorded numbers and relabeling them. This also serves as this gate's live "leakage
# re-confirmed" check on every upstream metric Gate 3 used (each is re-read/re-computed live here,
# not trusted from memory).
# ============================================================
print("\n" + "=" * 70)
print("SECTION 6: INDEPENDENT REPRODUCTION OF GATE 3'S CHAMPION SELECTION")
print("=" * 70)

correlation_result = compute_bp2_bp3_correlation(gold_pl)
reproduced_cramers_v = correlation_result["chi_square_cramers_v"]["cramers_v"]

upstream_metrics = load_upstream_validated_metrics(PROJECT_ROOT)
reproduced_bp4_coverage = compute_bp4_join_coverage(gold_pl)

friction_ordinal_ranks = load_bp2_friction_ordinal_ranks(PROJECT_ROOT)
signal_lazy = attach_normalized_signal_columns(gold_pl.lazy(), friction_ordinal_ranks)

raw_weights_by_candidate = compute_candidate_raw_weights(
    upstream_metrics, reproduced_bp4_coverage, reproduced_cramers_v
)
normalized_weights_by_candidate = {
    name: normalize_candidate_weights(w) for name, w in raw_weights_by_candidate.items()
}

reproduced_lr_diagnostic = fit_lr_diagnostic(gold_pl, sample_size=200_000, random_state=RANDOM_STATE)

reproduced_benchmark_rows: dict[str, dict] = {}
reproduced_scored_by_candidate: dict[str, pl.DataFrame] = {}
for name in CANDIDATE_NAMES:
    scored_candidate_pl = score_priority_rule(
        signal_lazy, normalized_weights_by_candidate[name], DEFAULT_INTERVENTION_THRESHOLD
    ).collect()
    row = benchmark_candidate(
        scored_candidate_pl, name, normalized_weights_by_candidate[name], raw_weights_by_candidate[name]
    )
    if name == "correlation_aware_plus_lr_diagnostic":
        row["lr_diagnostic_held_out_roc_auc"] = reproduced_lr_diagnostic["held_out_roc_auc"]
        row["lr_diagnostic_n_rows_fit_sample"] = reproduced_lr_diagnostic["n_rows_fit_sample"]
    reproduced_benchmark_rows[name] = row
    reproduced_scored_by_candidate[name] = scored_candidate_pl

REPRODUCED_CHAMPION = select_champion(reproduced_benchmark_rows, reproduced_cramers_v)
champion_row = reproduced_benchmark_rows[REPRODUCED_CHAMPION]
champion_weights_normalized = normalized_weights_by_candidate[REPRODUCED_CHAMPION]
scored_pl = reproduced_scored_by_candidate[REPRODUCED_CHAMPION]
champion_redundancy_score = reproduced_cramers_v * (
    champion_row["weight_bp2_normalized"] + champion_row["weight_bp3_normalized"]
)

print(f"[REPRODUCED] champion_rule_scheme = {REPRODUCED_CHAMPION!r} (Gate 3 recorded: {RECORDED_CHAMPION!r})")
print(
    f"[REPRODUCED] weights: bp2={champion_weights_normalized['bp2']:.6f}, "
    f"bp3={champion_weights_normalized['bp3']:.6f}, bp4={champion_weights_normalized['bp4']:.6f}"
)
print(
    f"[REPRODUCED] cramers_v={reproduced_cramers_v:.6f} (config recorded: "
    f"{full_config.get('bp2_bp3_cramers_v')}), bp4_coverage={reproduced_bp4_coverage:.6f} "
    f"(config recorded: {full_config.get('bp4_join_coverage')})"
)
print(
    f"[REPRODUCED] coverage_pct={champion_row['coverage_pct']} (config recorded: "
    f"{full_config.get('champion_coverage_pct')}), bp3_agreement_rate="
    f"{champion_row['bp3_agreement_rate']} (config recorded: "
    f"{full_config.get('champion_bp3_agreement_rate')})"
)

TOLERANCE = 1e-6


def _close(a, b, tol: float = TOLERANCE) -> bool:
    if a is None or b is None:
        return a == b
    return abs(float(a) - float(b)) < tol


champion_name_matches = REPRODUCED_CHAMPION == RECORDED_CHAMPION
weights_match = (
    _close(champion_weights_normalized["bp2"], full_config.get("champion_weight_bp2"))
    and _close(champion_weights_normalized["bp3"], full_config.get("champion_weight_bp3"))
    and _close(champion_weights_normalized["bp4"], full_config.get("champion_weight_bp4"))
)
cramers_v_matches = _close(reproduced_cramers_v, full_config.get("bp2_bp3_cramers_v"), tol=1e-4)
bp4_coverage_matches = _close(reproduced_bp4_coverage, full_config.get("bp4_join_coverage"), tol=1e-4)
coverage_pct_matches = _close(
    champion_row["coverage_pct"], full_config.get("champion_coverage_pct"), tol=1e-3
)
bp3_agreement_matches = _close(
    champion_row["bp3_agreement_rate"], full_config.get("champion_bp3_agreement_rate"), tol=1e-4
)
redundancy_score_matches = _close(
    champion_redundancy_score, full_config.get("champion_redundancy_double_counting_score"), tol=1e-4
)
reproduction_bit_exact = bool(
    champion_name_matches
    and weights_match
    and cramers_v_matches
    and bp4_coverage_matches
    and coverage_pct_matches
    and bp3_agreement_matches
    and redundancy_score_matches
)
print(f"\n[CHECK] Reproduction matches Gate 3's recorded results: {reproduction_bit_exact}")

reproduction_check = {
    "bp_id": "bp7",
    "gate": 4,
    "reproduced_champion_rule_scheme": REPRODUCED_CHAMPION,
    "gate3_recorded_champion_rule_scheme": RECORDED_CHAMPION,
    "champion_name_matches": champion_name_matches,
    "reproduced_champion_weights_normalized": champion_weights_normalized,
    "gate3_recorded_champion_weights_normalized": {
        "bp2": full_config.get("champion_weight_bp2"),
        "bp3": full_config.get("champion_weight_bp3"),
        "bp4": full_config.get("champion_weight_bp4"),
    },
    "weights_match": weights_match,
    "reproduced_bp2_bp3_cramers_v": reproduced_cramers_v,
    "gate3_recorded_bp2_bp3_cramers_v": full_config.get("bp2_bp3_cramers_v"),
    "cramers_v_matches": cramers_v_matches,
    "reproduced_bp4_join_coverage": reproduced_bp4_coverage,
    "gate3_recorded_bp4_join_coverage": full_config.get("bp4_join_coverage"),
    "bp4_coverage_matches": bp4_coverage_matches,
    "reproduced_champion_coverage_pct": champion_row["coverage_pct"],
    "gate3_recorded_champion_coverage_pct": full_config.get("champion_coverage_pct"),
    "coverage_pct_matches": coverage_pct_matches,
    "reproduced_champion_bp3_agreement_rate": champion_row["bp3_agreement_rate"],
    "gate3_recorded_champion_bp3_agreement_rate": full_config.get("champion_bp3_agreement_rate"),
    "bp3_agreement_matches": bp3_agreement_matches,
    "reproduced_champion_redundancy_double_counting_score": champion_redundancy_score,
    "gate3_recorded_champion_redundancy_double_counting_score": full_config.get(
        "champion_redundancy_double_counting_score"
    ),
    "redundancy_score_matches": redundancy_score_matches,
    "reproduction_bit_exact_within_tolerance": reproduction_bit_exact,
    "tolerance": TOLERANCE,
    "disclosure": (
        "Independent reproduction of Gate 3's entire champion-selection pipeline, re-derived "
        "from scratch in this gate's own fresh run against the identical real Gold layer, reusing "
        "Gate 3's own functions unmodified - never read from Gate 3's own config-recorded numbers "
        "and relabeled. Maps onto the Master Plan's generic Gate 4 'calibration' output slot, "
        "adapted the same way BP6 Gate 4 adapted it into a fresh-kernel bit-exact reproduction."
    ),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
with open(REPRODUCTION_CHECK_PATH, "w", encoding="utf-8") as f:
    json.dump(reproduction_check, f, indent=2, default=str)
print(f"[SAVED] {REPRODUCTION_CHECK_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 7: BOOTSTRAP CI ("Bootstrap CI" output) - real percentile bootstrap 95% CI on two real
# champion-output statistics: intervention_flag_rate and bp3_agreement_rate.
# ============================================================
print("\n" + "=" * 70)
print(f"SECTION 7: BOOTSTRAP CI ({N_DEFAULT_BOOTSTRAP} resamples, seed={RANDOM_STATE})")
print("=" * 70)

intervention_flag_array = scored_pl["intervention_flag"].to_numpy().astype(np.float64)
intervention_flag_rate_ci = bootstrap_ci_for_rate(
    intervention_flag_array, n_bootstrap=N_DEFAULT_BOOTSTRAP, random_state=RANDOM_STATE
)
print(
    f"[RESULT] intervention_flag_rate: point={intervention_flag_rate_ci['point_estimate']}, "
    f"95% CI=[{intervention_flag_rate_ci['ci_lower_95']}, {intervention_flag_rate_ci['ci_upper_95']}]"
)

bp3_compared_pl = scored_pl.filter(pl.col("bp3_predicted_label").is_not_null())
agree_array = (
    (bp3_compared_pl["intervention_flag"].to_numpy())
    == (bp3_compared_pl["bp3_predicted_label"].to_numpy() == 1)
).astype(np.float64)
bp3_agreement_rate_ci = bootstrap_ci_for_rate(
    agree_array, n_bootstrap=N_DEFAULT_BOOTSTRAP, random_state=RANDOM_STATE
)
print(
    f"[RESULT] bp3_agreement_rate: point={bp3_agreement_rate_ci['point_estimate']}, "
    f"95% CI=[{bp3_agreement_rate_ci['ci_lower_95']}, {bp3_agreement_rate_ci['ci_upper_95']}]"
)

bootstrap_ci_output = {
    "bp_id": "bp7",
    "gate": 4,
    "champion_rule_scheme": REPRODUCED_CHAMPION,
    "intervention_flag_rate": intervention_flag_rate_ci,
    "bp3_agreement_rate": bp3_agreement_rate_ci,
    "disclosure": (
        "Real percentile bootstrap 95% CIs (resampling with replacement, at the real input array's "
        "own size, disclosed seed and resample count) - reported as a range, never a point "
        "estimate presented as exact. Maps onto the Master Plan's generic Gate 4 'Bootstrap CI' "
        "output slot."
    ),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
with open(BOOTSTRAP_CI_PATH, "w", encoding="utf-8") as f:
    json.dump(bootstrap_ci_output, f, indent=2, default=str)
print(f"[SAVED] {BOOTSTRAP_CI_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 8: CONFUSION-MATRIX ANALOG ("confusion matrix" output) - real 2x2 agreement/reference
# cross-tab of intervention_flag x bp3_predicted_label, honestly labeled (never a true confusion
# matrix - no ground-truth column exists for customer360_priority_decision).
# ============================================================
print("\n" + "=" * 70)
print("SECTION 8: CONFUSION-MATRIX ANALOG (agreement/reference cross-tab vs BP3's own prediction)")
print("=" * 70)

agreement_crosstab = build_bp3_agreement_crosstab(scored_pl)
print(f"[RESULT] {agreement_crosstab}")
with open(AGREEMENT_CROSSTAB_PATH, "w", encoding="utf-8") as f:
    json.dump(agreement_crosstab, f, indent=2, default=str)
print(f"[SAVED] {AGREEMENT_CROSSTAB_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: EXPLAINABILITY ("SHAP sample" output) - exact per-row contribution decomposition of
# priority_score into its three additive terms. A genuine strength over SHAP here: this is an
# EXACT decomposition of a known, transparent formula, not an approximation.
# ============================================================
print("\n" + "=" * 70)
print("SECTION 9: EXPLAINABILITY - EXACT PRIORITY_SCORE CONTRIBUTION DECOMPOSITION")
print("=" * 70)

decomp_pl = compute_priority_score_contribution_decomposition(scored_pl, champion_weights_normalized)
contribution_summary = summarize_contribution_decomposition(decomp_pl)
print(
    f"[RESULT] mean contributions: bp2={contribution_summary['mean_contribution_bp2']}, "
    f"bp3={contribution_summary['mean_contribution_bp3']}, "
    f"bp4={contribution_summary['mean_contribution_bp4']}"
)
print(
    f"[CHECK] max_abs_reconstruction_error={contribution_summary['max_abs_reconstruction_error']} "
    f"(exact within tolerance: {contribution_summary['reconstruction_exact_within_tolerance']})"
)
with open(CONTRIBUTION_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(contribution_summary, f, indent=2, default=str)
print(f"[SAVED] {CONTRIBUTION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")

CONTRIBUTION_SAMPLE_SIZE = min(50, decomp_pl.height)
contribution_sample_pl = decomp_pl.filter(pl.col("priority_score").is_not_null()).sample(
    n=min(CONTRIBUTION_SAMPLE_SIZE, decomp_pl.filter(pl.col("priority_score").is_not_null()).height),
    seed=RANDOM_STATE,
)
contribution_sample_pl.write_csv(CONTRIBUTION_SAMPLE_CSV_PATH)
print(
    f"[SAVED] {CONTRIBUTION_SAMPLE_CSV_PATH.relative_to(PROJECT_ROOT)} "
    f"({contribution_sample_pl.height} bounded sample rows, seed={RANDOM_STATE})"
)

# ============================================================
# SECTION 10: LEAKAGE RE-CONFIRMED - fresh, live re-check that no barred column is present on the
# real scoring Gold layer, never trusted from Gate 1/2/3's own prior claim.
# ============================================================
print("\n" + "=" * 70)
print("SECTION 10: LEAKAGE RE-CONFIRMATION (fresh, live)")
print("=" * 70)

leakage_reconfirmation = reconfirm_no_barred_columns_in_gold_layer(gold_pl.columns)
print(f"[RESULT] leakage_reconfirmed_clean = {leakage_reconfirmation['leakage_reconfirmed_clean']}")
for col, present in leakage_reconfirmation["presence_in_scoring_gold_layer"].items():
    print(f"    {col}: present={present}")
with open(LEAKAGE_RECONFIRMATION_PATH, "w", encoding="utf-8") as f:
    json.dump(leakage_reconfirmation, f, indent=2, default=str)
print(f"[SAVED] {LEAKAGE_RECONFIRMATION_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 11: DISPARATE-IMPACT / ECOA REG B AUDIT (the real substance of this gate's compliance
# requirement) - resolves Gate 3's honest deferral for real. See this notebook's own markdown cell
# for the full real investigation this design follows.
# ============================================================
print("\n" + "=" * 70)
print("SECTION 11: DISPARATE-IMPACT / ECOA REG B AUDIT-ONLY CHECK")
print("=" * 70)

with open(BP7_POLICY_PATH, "r", encoding="utf-8") as f:
    bp7_policy = json.load(f)
ecoa_reg_b_text = bp7_policy.get("compliance_touchpoint", {}).get("ecoa_reg_b", "")
gate1_commits_to_gate4_check = "own Gate 4" in ecoa_reg_b_text and "tags_group" in ecoa_reg_b_text
print("[LIVE-READ] Gate 1 policy.json compliance_touchpoint.ecoa_reg_b (first 200 chars):")
print(f"    {ecoa_reg_b_text[:200]}...")
print(
    f"[CHECK] Gate 1's own text commits to this exact Gate 4 tags_group check: {gate1_commits_to_gate4_check}"
)

if not BP3_GOLD_PATH.exists():
    raise FileNotFoundError(
        f"{BP3_GOLD_PATH} not found - BP3's own real Gold layer is required to resolve Gate 3's "
        "own honestly-deferred disparate-impact carry-forward check (see this notebook's markdown "
        "cell for the full investigation). This is a genuine blocker, not worked around here: if "
        "this file is truly absent, this check must be honestly re-deferred, never fabricated."
    )

tags_group_pl = load_bp3_gold_tags_group(BP3_GOLD_PATH)
print(
    f"[OK] Loaded real tags_group from BP3's own Gold layer: {tags_group_pl.height:,} rows "
    f"(Complaint ID + tags_group, read-only, audit-only)."
)
tags_group_distribution = tags_group_pl["tags_group"].value_counts().sort("tags_group").to_dicts()
print(f"[LIVE] Real tags_group distribution (from BP3's own Gold layer): {tags_group_distribution}")

audited_pl = attach_tags_group_for_audit(scored_pl, tags_group_pl)
disparate_impact_result = compute_disparate_impact_audit(audited_pl)
print(
    f"[RESULT] join_coverage_pct={disparate_impact_result['join_coverage_pct']}, "
    f"adverse_impact_ratio={disparate_impact_result['adverse_impact_ratio']}, "
    f"flagged_four_fifths_rule={disparate_impact_result['flagged_four_fifths_rule']}"
)
for row in disparate_impact_result["group_breakdown"]:
    print(f"    {row}")

with open(BP3_GATE4_STAT_VALIDATION_PATH, "r", encoding="utf-8") as f:
    bp3_gate4_stat_validation = json.load(f)
bp3_own_adverse_impact_ratio = bp3_gate4_stat_validation.get("adverse_impact_ratio_tags")
print(
    f"\n[COMPARISON] BP3's own real, already-closed Gate 4 adverse_impact_ratio (on BP3's own "
    f"intervention prediction) = {bp3_own_adverse_impact_ratio} vs BP7's own champion "
    f"intervention_flag adverse_impact_ratio (this gate, real) = "
    f"{disparate_impact_result['adverse_impact_ratio']}. These are two DIFFERENT real decisions "
    "(BP3's own model prediction vs BP7's own downstream rule-based intervention_flag) measured "
    "with the identical real statistical convention - not expected to be numerically equal, both "
    "reported for governance context."
)

disparate_impact_result["data_source"] = (
    "BP3's own real, already-governance-reviewed Gold layer "
    "(data/processed/cfpb_intervention_escalation_gold.parquet), Complaint-ID-keyed Tags "
    "passthrough column, read-only, never a BP7 scoring input - joined onto BP7's own "
    "already-scored population strictly after priority_score/intervention_flag/recommended_action "
    "were computed."
)
disparate_impact_result["tags_group_distribution_source"] = tags_group_distribution
disparate_impact_result["bp3_own_gate4_adverse_impact_ratio_tags_for_comparison"] = (
    bp3_own_adverse_impact_ratio
)
disparate_impact_result["gate1_policy_commits_to_this_check"] = gate1_commits_to_gate4_check
disparate_impact_result["resolution_of_gate3_deferral"] = (
    "Gate 3 honestly deferred this check because BP7's own Gate 2 Gold layer does not preserve "
    "tags_group and reconstructing it from the raw, barred 'Tags' column would have been a Gate 3 "
    "SCORING input, which Gate 1's leakage_rules bars. This gate resolves that deferral by reading "
    "tags_group from BP3's OWN separate, already-governance-approved Gold layer (never the raw "
    "CFPB extract directly, never any BP1-6 file modified) and joining it strictly AFTER scoring, "
    "for audit-grouping only - the same input-vs-audit-only distinction BP3's own Gate 4 and BP7 "
    "Gate 1's own policy.json text both already drew."
)
with open(DISPARATE_IMPACT_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(disparate_impact_result, f, indent=2, default=str)
print(f"[SAVED] {DISPARATE_IMPACT_AUDIT_PATH.relative_to(PROJECT_ROOT)}")

pd.DataFrame(disparate_impact_result["group_breakdown"]).to_csv(
    DISPARATE_IMPACT_BREAKDOWN_CSV_PATH, index=False
)
print(f"[SAVED] {DISPARATE_IMPACT_BREAKDOWN_CSV_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 12: Write the Gate 4 master summary JSON artifact.
# ============================================================
gate4_summary = {
    "bp_id": "bp7",
    "gate": 4,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "live_row_count": live_row_count,
    "champion_rule_scheme": REPRODUCED_CHAMPION,
    "reproduction_check": {
        "reproduction_bit_exact_within_tolerance": reproduction_bit_exact,
        "path": str(REPRODUCTION_CHECK_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
    "bootstrap_ci": {
        "intervention_flag_rate": intervention_flag_rate_ci,
        "bp3_agreement_rate": bp3_agreement_rate_ci,
        "path": str(BOOTSTRAP_CI_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
    "bp3_agreement_crosstab": {
        "agreement_rate": agreement_crosstab["agreement_rate"],
        "path": str(AGREEMENT_CROSSTAB_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
    "contribution_decomposition": {
        "max_abs_reconstruction_error": contribution_summary["max_abs_reconstruction_error"],
        "reconstruction_exact_within_tolerance": contribution_summary[
            "reconstruction_exact_within_tolerance"
        ],
        "summary_path": str(CONTRIBUTION_SUMMARY_PATH.relative_to(PROJECT_ROOT).as_posix()),
        "sample_path": str(CONTRIBUTION_SAMPLE_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
    "leakage_reconfirmation": {
        "leakage_reconfirmed_clean": leakage_reconfirmation["leakage_reconfirmed_clean"],
        "path": str(LEAKAGE_RECONFIRMATION_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
    "disparate_impact_audit": {
        "performed": True,
        "deferred_at_gate3_now_resolved": True,
        "join_coverage_pct": disparate_impact_result["join_coverage_pct"],
        "adverse_impact_ratio": disparate_impact_result["adverse_impact_ratio"],
        "flagged_four_fifths_rule": disparate_impact_result["flagged_four_fifths_rule"],
        "audit_path": str(DISPARATE_IMPACT_AUDIT_PATH.relative_to(PROJECT_ROOT).as_posix()),
        "breakdown_path": str(DISPARATE_IMPACT_BREAKDOWN_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
}
with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(gate4_summary, f, indent=2, default=str)
print(f"\n[SAVED] {SUMMARY_JSON_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 13: Write the Gate 4 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified). `status` is deliberately NOT touched here - this project's own
# established convention (confirmed at BP7 Gate 2 and Gate 3 both) is that `status` is updated
# only at a BP's own final gate.
# ============================================================
gate4_marker = (
    "# --- Gate 4 (Statistical Validation & Explainability) results (appended, idempotent overwrite) ---"
)
gate4_block_lines = [
    f"gate4_champion_reproduced_bit_exact: {reproduction_bit_exact}",
    f"gate4_reproduced_intervention_flag_rate: {intervention_flag_rate_ci['point_estimate']}",
    f"gate4_intervention_flag_rate_bootstrap_ci_95: "
    f"[{intervention_flag_rate_ci['ci_lower_95']}, {intervention_flag_rate_ci['ci_upper_95']}]",
    f"gate4_bp3_agreement_rate_point_estimate: {bp3_agreement_rate_ci['point_estimate']}",
    f"gate4_bp3_agreement_rate_bootstrap_ci_95: "
    f"[{bp3_agreement_rate_ci['ci_lower_95']}, {bp3_agreement_rate_ci['ci_upper_95']}]",
    f"gate4_bootstrap_n_resamples: {N_DEFAULT_BOOTSTRAP}",
    f"gate4_contribution_decomposition_max_abs_reconstruction_error: "
    f"{contribution_summary['max_abs_reconstruction_error']}",
    f"gate4_leakage_reconfirmed_clean: {leakage_reconfirmation['leakage_reconfirmed_clean']}",
    "gate4_disparate_impact_check_performed: True",
    "gate4_disparate_impact_deferred_at_gate3_now_resolved: True",
    f"gate4_disparate_impact_join_coverage_pct: {disparate_impact_result['join_coverage_pct']}",
    f"gate4_disparate_impact_adverse_impact_ratio: {disparate_impact_result['adverse_impact_ratio']}",
    f"gate4_disparate_impact_flagged_four_fifths_rule: {disparate_impact_result['flagged_four_fifths_rule']}",
    f"gate4_disparate_impact_lowest_selection_rate_group: "
    f'"{disparate_impact_result["lowest_selection_rate_group"]}"',
    f"gate4_disparate_impact_highest_selection_rate_group: "
    f'"{disparate_impact_result["highest_selection_rate_group"]}"',
    f"gate4_disparate_impact_bp3_own_adverse_impact_ratio_for_comparison: {bp3_own_adverse_impact_ratio}",
    f'gate4_reproduction_check_path: "{REPRODUCTION_CHECK_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'gate4_bootstrap_ci_path: "{BOOTSTRAP_CI_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'gate4_bp3_agreement_crosstab_path: "{AGREEMENT_CROSSTAB_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f"gate4_contribution_decomposition_summary_path: "
    f'"{CONTRIBUTION_SUMMARY_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f"gate4_leakage_reconfirmation_path: "
    f'"{LEAKAGE_RECONFIRMATION_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f"gate4_disparate_impact_audit_path: "
    f'"{DISPARATE_IMPACT_AUDIT_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'gate4_summary_path: "{SUMMARY_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
]
write_gate_block(BP7_CONFIG_PATH, gate4_marker, gate4_block_lines)
print(f"[SAVED] gate4 block written to {BP7_CONFIG_PATH.relative_to(PROJECT_ROOT)} (status untouched)")

with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    _post_write_config_text = f.read()
gate4_block_actually_written = gate4_marker in _post_write_config_text
_post_write_config = yaml.safe_load(_post_write_config_text)
status_untouched = _post_write_config.get("status") == full_config.get("status")

# ============================================================
# SECTION 14: Structural integrity checks - raise AssertionError, never silently pass.
# ============================================================
checks = {
    "gate3_prerequisite_confirmed": gate3_confirmed,
    "gold_layer_row_count_matches_config": row_count_matches_config,
    "champion_reproduced_bit_exact_within_tolerance": reproduction_bit_exact,
    "bootstrap_ci_intervention_flag_rate_computed": (
        intervention_flag_rate_ci["ci_lower_95"] <= intervention_flag_rate_ci["ci_upper_95"]
    ),
    "bootstrap_ci_bp3_agreement_rate_computed": (
        bp3_agreement_rate_ci["ci_lower_95"] <= bp3_agreement_rate_ci["ci_upper_95"]
    ),
    "agreement_crosstab_n_compared_positive": agreement_crosstab["n_compared"] > 0,
    "contribution_decomposition_reconstruction_exact": contribution_summary[
        "reconstruction_exact_within_tolerance"
    ],
    "contribution_sample_csv_written": CONTRIBUTION_SAMPLE_CSV_PATH.exists(),
    "leakage_reconfirmed_clean_on_scoring_gold_layer": leakage_reconfirmation["leakage_reconfirmed_clean"],
    "disparate_impact_join_coverage_is_100_pct": disparate_impact_result["join_coverage_pct"] == 100.0,
    "disparate_impact_adverse_impact_ratio_in_valid_range": (
        disparate_impact_result["adverse_impact_ratio"] is not None
        and 0.0 <= disparate_impact_result["adverse_impact_ratio"] <= 1.0
    ),
    "disparate_impact_all_four_known_groups_present": (
        len(disparate_impact_result["group_breakdown"]) == len(KNOWN_TAGS_GROUPS)
    ),
    "gate1_policy_text_confirms_this_gate4_commitment": gate1_commits_to_gate4_check,
    "reproduction_check_json_written": REPRODUCTION_CHECK_PATH.exists(),
    "bootstrap_ci_json_written": BOOTSTRAP_CI_PATH.exists(),
    "agreement_crosstab_json_written": AGREEMENT_CROSSTAB_PATH.exists(),
    "contribution_summary_json_written": CONTRIBUTION_SUMMARY_PATH.exists(),
    "leakage_reconfirmation_json_written": LEAKAGE_RECONFIRMATION_PATH.exists(),
    "disparate_impact_audit_json_written": DISPARATE_IMPACT_AUDIT_PATH.exists(),
    "disparate_impact_breakdown_csv_written": DISPARATE_IMPACT_BREAKDOWN_CSV_PATH.exists(),
    "gate4_summary_json_written": SUMMARY_JSON_PATH.exists(),
    "config_gate4_block_written": gate4_block_actually_written,
    "status_field_untouched": status_untouched,
}

print("\n=== INTEGRITY CHECKS ===")
for check_name, passed in checks.items():
    result_label = "[PASS]" if passed else "[FAIL]"
    print(f"{result_label} {check_name}")
    assert passed, f"[CHECK FAILED] {check_name}"

print(
    f"\n[ALL CHECKS PASSED] BP7 Gate 4 complete - champion='{REPRODUCED_CHAMPION}' reproduced "
    f"bit-exact={reproduction_bit_exact}. Disparate-impact/ECOA Reg B audit: adverse_impact_ratio="
    f"{disparate_impact_result['adverse_impact_ratio']}, flagged="
    f"{disparate_impact_result['flagged_four_fifths_rule']} (Gate 3's own honest deferral resolved "
    "for real, not worked around). Proceed to BP7 Gate 5 next."
)